# 🫀 퀘스트 46 · Q7-V — **자가 병목인가**: 검출기 오차를 실측해 주입한다

| | **MedKOS / `notebooks/quest46_q7v_ruler_audit.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | **Q7-U**(자 세우기 · Se 0.9109/PPV 0.7145) · **Q7-P0**(SVDB P 표) · **Q7-S′** |
| 규약 | **R11 · R16 · R22 · R29 ② · R33 ① · R34 ③ · R35 ①④⑦ · R36 ①④** |
| 학습 | **0회** · GPU 불필요 · 새 데이터 수집 **0** · 예상 25~40분 |

## 이 런이 답하는 질문 하나

Q7-S′ 의 S3 는 초과 **+0.0272**, MDE **0.0531** 로 미결이었다. 그다음 결정은
**「데이터를 더 모을 것인가」**인데, 그 답은 딱 하나에 달려 있다.

> **자가 무뎌서 못 재는 것이라면, 437 레코드를 모아도 안 올라온다.**

우리 자는 이렇게 생겼다(Q7-U 실측, BUT PDB 전문가 주석 대비):

```
Se  0.9109   ← 정답 P 의 91% 를 잡는다
PPV 0.7145   ← ★ 발화의 **28.55% 가 틀린 자리**다
```

**PPV 0.7145 는 「조용한 오검출」이 가설이 아니라 측정된 사실**이라는 뜻이다.
검출기는 기권하지 않으므로, 틀릴 때 「없음」이 아니라 **엉뚱한 위치의 값**을 준다.
그 값이 그대로 `p_score`·`pr_q8` 가 됐다. Q7-S′ 는 그걸 알면서 돌렸다.

## 방법 — 두 갈래로 같은 수를 구한다

### ★★★ V1 (주 관문) — **감쇠 상한**

BUT PDB 에는 정답이 있으므로, **같은 비트에서** 특징을 두 번 잴 수 있다.

```
x_true = feature(정답 P 위치)      x_obs = feature(검출 위치)
λ = corr(x_true, x_obs)            ← **감쇠 계수**
```

고전 측정오차 이론: 관측된 표준화 효과는 참값의 **λ배**로 줄어든다. 그래서

```
d_obs  = √2 · Φ⁻¹(AUROC_obs)
d_true = d_obs / λ
AUROC_true = Φ(d_true / √2)        ← ★ **자를 완벽히 고쳤을 때의 상한**
```

⚠️ **이건 점추정이 아니라 상한이다.** 탈감쇠는 **한 방향으로만** 움직인다
(항상 키운다). 그래서 판정이 깨끗해진다:

- **상한조차 MDE 밑** → **자를 고쳐도 못 넘는다.** 신호가 진짜로 이 크기다
  → 자가 병목이 **아니다**. 남은 건 표본 문제
- **상한이 MDE 위** → 자가 병목**일 수 있다** → 데이터를 모으기 전에 자부터 고친다

### ★★ V2 — **용량-반응**(다른 계산)

SVDB 에는 정답이 없으니 「지터를 빼는」건 불가능하다. 대신 **측정된 오차를 한 번 더
얹는다**(k = 0, 1, 2). 기울기가 답이다.

- 한 번 더 얹었는데 **초과분이 무너지면** → 지금 이미 한 번 얹혀 있고, 그게 신호를
  깎고 있다는 뜻이다(자가 병목 쪽)
- 한 번 더 얹어도 **평평하면** → 측정이 이 오차에 둔감하다(신호 크기 쪽)

★ **k=0 은 항등이어야 한다** — 오차를 0번 얹은 결과는 Q7-S′ 의 S3 와 **정확히 같은
값**이어야 한다. 다르면 주입 파이프라인이 틀린 것이므로 **중단**한다(R35 ④).

⚠️ V1 과 V2 는 **같은 실측 쌍 분포를 공유한다** — 「독립 경로」가 아니라 **「닫힌형
이론 상한」 vs 「그 분포를 실제 추정량(리듬 잔차화·레코드 구조·매칭)에 통과시킨
재표집」**이다. 둘이 2배 넘게 어긋나면 어느 쪽도 믿지 않고 그렇게 적는다.

### V3 — **오검출 해부**

틀린 발화는 **어디에** 떨어지나. 직전 T 인가, QRS 개시인가, 기저선인가.
「조용한 오검출」의 정체를 그림으로 박는다. 이게 다음에 무슨 자를 써야 하는지를 정한다.

## 사전등록 — 관문

| 관문 | 내용 | 판정 |
|---|---|---|
| **V0** | **재현 증명** — Q7-U 의 `dwt\|raw` Se·PPV·F1 을 다시 낸다 | 크게 어긋나면 **중단** |
| **V1 ★★★(주)** | **감쇠 상한** — 자를 완벽히 고쳤을 때 S3 가 갈 수 있는 최대 | 상한 초과 vs MDE |
| **V2 ★★** | **용량-반응** — 오차를 더 얹으면 초과분이 무너지나 | k=0 항등 필수 |
| **V3** | 오검출 해부 — 틀린 발화가 떨어지는 자리 | 기술통계 |
| **V4** | **영점** — 같은 주입을 무정보 특징에 걸면 0 을 유지하나 | 0 근처여야 |

### 판정표 (R29 ②)

- **V0 실패** → 자가 그때 그 자가 아니다. **어떤 관문도 읽지 않는다**
- **V2 의 k=0 이 항등이 아니다** → 주입이 틀렸다. **중단**
- **V1 상한 ≤ MDE** → ★ **자는 병목이 아니다.** 「자를 더 갈면 된다」는 길을 닫는다
  → 남은 선택은 **코호트를 키우거나 갈래를 접는 것**뿐
- **V1 상한 > MDE 이고 V2 기울기가 가파르다** → ★ **자가 병목이다.**
  데이터 수집을 **보류**하고 분절기부터 고친다
- **V1 과 V2 가 2배 넘게 어긋난다** → 둘 다 인용하지 않고 그렇게 적는다
- ⚠️ 둘은 **같은 실측 쌍 분포를 공유**한다 — 일치해도 **독립 확인이 아니다**

⚠️ **이 런은 SVEB 질문에 답하지 않는다.** 「더 모을 가치가 있나」에만 답한다.

In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    """**레코드 단위** 부트스트랩(R11)."""
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def match_1d(det, ref, tol):
    """★ 탐욕적 1:1 — 가까운 쌍부터 확정하고 **양쪽 다 소모**한다.
    (같은 정답에 여러 발화가 붙어 Se 가 부풀지 않게)"""
    det = np.asarray(det, int); ref = np.asarray(ref, int)
    if not len(det) or not len(ref):
        return 0, np.array([], float)
    pairs = []
    for i, d in enumerate(det):
        j = int(np.argmin(np.abs(ref - d)))
        if abs(ref[j] - d) <= tol:
            pairs.append((abs(ref[j] - d), i, j))
    pairs.sort()
    ud, ur, err = set(), set(), []
    for e, i, j in pairs:
        if i in ud or j in ur:
            continue
        ud.add(i); ur.add(j); err.append(float(det[i] - ref[j]))
    return len(err), np.asarray(err, float)

def auc_to_d(a):
    """AUROC → 표준화 효과 d (이분정규 가정)."""
    from scipy.stats import norm
    a = float(np.clip(a, 1e-6, 1 - 1e-6))
    return float(np.sqrt(2.0) * norm.ppf(a))

def d_to_auc(d):
    from scipy.stats import norm
    return float(norm.cdf(d / np.sqrt(2.0)))

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260804, 1
FS, RPRE, L = 360, 100, 300

# ── ★ Q7-U 와 **같은** 자 (여기를 바꾸면 감사 대상이 달라진다)
BUT_DIR = "but-pdb/1.0.0"
DELIN   = "dwt"                              # Q7-U 의 U1 승자
INPUT   = "raw"                              # U2 가 소거를 검출 경로에서 내렸다
FIRE_RATE = 1.0                              # U1 이 보고된 작동점
TOL_MS  = 50.0                               # P 위치 허용 오차(문헌 관행)
P_LO_MS, P_HI_MS = -278.0, -42.0             # P 탐색창(R 기준) — Q7-U·Q7-P0 와 동일
SCORE_HALF_MS = 100.0                        # ★ Q7-P0 의 점수 창(위치 중심 ±)
MIN_P   = 5                                  # R17

# ── ★ Q7-U 공식 실행 `20260804T0458` 값 — V0 재현 증명 기준
REF_U1 = dict(se=0.9109, pp=0.7145, f1=0.7705)
TOL_U_WARN, TOL_U_FAIL = 0.02, 0.05          # 경고 / 중단

# ── SVDB 쪽 (Q7-S′ 와 동일 상수 — 안 맞으면 항등 대조가 깨진다)
QUANT_MS = 1000.0 / 128.0
FULL_K = tuple(range(4, 33))
MIN_S, MIN_N, MIN_PAIR = 25, 25, 200
DOSES = (0, 1, 2)                            # ★ 오차를 몇 번 더 얹나
N_REP = 8                                    # 주입 반복(잡음 평균)

# ── ★ Q7-S′ 공식 실행 `20260804T0658` — V2 의 **항등 대조** 기준
REF_S3 = dict(p_score=0.5362, null=0.5090, mde=0.0531, n=34)
IDENT_TOL = 5e-4                             # k=0 이 이보다 어긋나면 **중단**

SV5  = os.path.join(MITBIH, "svdb_data5.npz")
PDEL = os.path.join(MITBIH, "svdb_pdelin.npz")

RULE_CHECK = {
    "R11 환자 단위":      "부트스트랩·판정 전부 **레코드 단위**",
    "R16 fallback 없음":  "BUT PDB 를 못 받으면 **중단** — 합성 오차분포로 대체 안 함",
    "R22 누수 없음":      "오차 분포는 **BUT PDB** 에서 재고, SVDB 라벨은 안 본다",
    "R29 ② 분기 금지":    "V0·항등이 깨지면 어떤 결론 분기도 타지 않는다",
    "R33 ① MDE":          "★ 판정 자체가 **상한 vs MDE** 비교다",
    "R34 ③ 대조 보장":    "★★ **k=0 은 구성상 항등** — 어긋나면 주입이 틀린 것",
    "R35 ① 자 먼저":      "★ 이 런은 **자를 감사**한다 — 자를 안 재고 결론을 못 낸다",
    "R35 ④ 항등 대조":    "★★ k=0 · 영점 팔 둘 다 항등/0 이 **구성으로 보장**",
    "R35 ⑦ 정합 증명":    "★ V0 — Q7-U 값을 다시 내지 못하면 감사 대상이 다른 자다",
    "R36 ① 상한":         "★★ **탈감쇠는 상한이다** — 점추정으로 인용 금지",
}

CONFIG = dict(
    exp="quest46_q7v_ruler_audit", quest="ailab-2026-0046", step="ruler-audit",
    parent_exp=["quest46_q7u_public_delineator", "quest46_q7p0_svdb_pdelin",
                "quest46_q7s2_p_aligned"],
    purpose=("**자가 병목인가.** Q7-S′ 의 S3 는 초과 +0.0272 · MDE 0.0531 로 미결이었고, "
             "다음 결정(데이터를 더 모을 것인가)은 이 하나에 달려 있다 — 자가 무뎌서 못 "
             "재는 것이라면 437 레코드를 모아도 안 올라온다. Q7-U 실측 PPV 0.7145 는 "
             "**발화의 28.55%가 틀린 자리**라는 뜻이고, 검출기가 기권하지 않으므로 그 값이 "
             "그대로 특징이 됐다. BUT PDB 정답으로 **같은 비트에서 특징을 두 번 재어** "
             "감쇠 계수 λ 를 실측하고, 탈감쇠로 **자를 완벽히 고쳤을 때의 상한**을 낸다. "
             "독립 경로로 오차를 SVDB 에 **더 얹어** 용량-반응도 잰다"),
    dataset=("BUT PDB 50×2분×2유도(전문가 P 주석) + SVDB 78레코드 184,499비트"),
    delineator=DELIN, input=INPUT, fire_rate=FIRE_RATE, tol_ms=TOL_MS,
    p_win_ms=[P_LO_MS, P_HI_MS], score_half_ms=SCORE_HALF_MS,
    doses=list(DOSES), n_rep=N_REP, ref_u1=REF_U1, ref_s3=REF_S3,
    ident_tol=IDENT_TOL, rule_check=RULE_CHECK,
    predictions={
        "V0": "**재현 증명** — Q7-U `dwt|raw` @발화율 1.00 의 Se 0.9109 · PPV 0.7145 · "
              f"F1 0.7705 를 다시 낸다. |Δ| > {TOL_U_FAIL} 면 **중단**(감사 대상이 다른 자다)",
        "V1": "★★★ **(주) 감쇠 상한.** BUT PDB 에서 같은 비트의 특징을 **정답 위치**와 "
              "**검출 위치**에서 각각 재어 λ = corr 를 구한다. 고전 측정오차 이론으로 "
              "`d_true = d_obs/λ` → `AUROC_true = Φ(d_true/√2)`. ★ 탈감쇠는 **한 방향**"
              "으로만 움직이므로 이건 **상한**이다. **상한조차 MDE 밑이면 자를 고쳐도 "
              "못 넘는다** = 자는 병목이 아니다",
        "V2": "★★ **용량-반응**(다른 계산). SVDB 에 측정된 오차를 k=0,1,2 번 **더** 얹고 "
              "S3 를 다시 잰다. ★ **k=0 은 구성상 항등** — Q7-S′ 의 0.5362 와 "
              f"|Δ| ≤ {IDENT_TOL} 이어야 하고 아니면 **중단**. 기울기가 가파르면 자가 "
              "병목 쪽, 평평하면 신호 크기 쪽. V1 과 2배 넘게 어긋나면 둘 다 안 쓴다",
        "V3": "**오검출 해부** — 허용 오차 밖 발화가 직전 T · QRS 개시 · 기저선 중 어디에 "
              "떨어지나. 「조용한 오검출」의 정체이고, 다음에 무슨 자를 써야 하는지를 정한다",
        "V4": "**영점** — 같은 주입을 무정보 특징(창 안 무작위 위치 + 무작위 점수)에 걸면 "
              "매칭 AUROC 가 0.5 를 유지해야 한다. 주입이 인공 신호를 만들지 않는다는 보증"},
    caveat=("★ **이 런은 SVEB 질문에 답하지 않는다** — 「더 모을 가치가 있나」에만 답한다. "
            "★ **탈감쇠는 상한이다**(R36 ①) — 점추정으로 인용하면 안 된다. 고전 측정오차 "
            "가정(가법 오차·이분정규·선형 판별)이 깨지면 상한이 더 느슨해질 뿐 방향은 같다. "
            "★ **오차 분포는 BUT PDB 것이다** — SVDB 와 코호트도(2분 vs 30분 · 부정맥 구성) "
            "**표본율도**(BUT PDB 원본 vs SVDB 128Hz) 다르다. 옮겨진 분포에는 전이 가정이 "
            "둘 붙고, 노트북이 실측 표본율을 찍어 그 크기를 보인다. 방향은 보수적이다 — "
            "해상도가 거칠수록 실제 감쇠는 더 심하고 V1 의 상한은 더 낮게 잡힌다. "
            "★ 학습 0회 · 예상 25~40분"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7v_ruler_audit", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q7-V 자 감사** — 자가 병목인가")
run.log(f"  자 = `{DELIN}|{INPUT}` @발화율 {FIRE_RATE:.2f} (Q7-U 승자) · "
        f"허용 오차 ±{TOL_MS:.0f}ms")
run.log(f"  ★★ 주 관문 V1 = **감쇠 상한** — 자를 완벽히 고쳤을 때 S3 가 갈 수 있는 최대")
run.log(f"  ★  독립 경로 V2 = 용량-반응 k={DOSES} · **k=0 은 항등**(허용 {IDENT_TOL})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")

In [ ]:
# CELL 2 — 【V-0】 BUT PDB 적재 (Q7-T 가 두 번 멈춘 자리 — 둘 다 실물로 확인)
try:
    import wfdb, neurokit2 as nk
except ImportError:
    !pip -q install wfdb neurokit2
    importlib.invalidate_caches(); import wfdb, neurokit2 as nk
import re, urllib.request

run.log("\n" + "=" * 100)
run.log("【V-0】 BUT PDB — 전문가 P 주석 (외부 정답)")
run.log("=" * 100)
run.log(f"  neurokit2 {nk.__version__} · wfdb {getattr(wfdb, '__version__', '?')}")
BASE = f"https://physionet.org/files/{BUT_DIR}"
BEAT_SYM = set("NLRAaJSVFejE/fQ")

def _get(url, timeout=60):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode("utf-8", "replace")

BUT_RECS = [str(r) for r in wfdb.get_record_list("but-pdb")]
if len(BUT_RECS) < 10:
    raise AssetError(f"BUT PDB 레코드 목록이 {len(BUT_RECS)}개 — 다운로드 실패(R16)")

def resolve_rid(r):
    """★ RECORDS 는 `1` 인데 파일은 `01.hea` 다 — 추측하지 않고 헤더로 확인한다."""
    for c in dict.fromkeys([r] + ([f"{int(r):0{w}d}" for w in (2, 3)] if r.isdigit() else [])):
        try:
            wfdb.rdheader(c, pn_dir=BUT_DIR); return c
        except Exception:
            continue
    return None

_res = [(r, resolve_rid(r)) for r in BUT_RECS]
BUT_RECS = [c for _, c in _res if c is not None]
if len(BUT_RECS) < 10:
    raise AssetError(f"헤더가 읽히는 레코드가 {len(BUT_RECS)}개")

def list_exts(rid):
    exts = []
    try:
        for ln in _get(f"{BASE}/ANNOTATORS").splitlines():
            tok = ln.split("\t")[0].strip() if ln.strip() else ""
            if tok and not tok.startswith("#"):
                exts.append(tok.split()[0])
    except Exception as e:
        run.log(f"  ⚠️ ANNOTATORS 를 못 읽었다: {type(e).__name__} {e}")
    if not exts:
        try:
            html = _get(BASE + "/")
            exts = sorted({x for x in re.findall(rf"{re.escape(rid)}\.([A-Za-z0-9_]+)", html)
                           if x not in ("dat", "hea", "xws", "png", "txt")})
        except Exception as e:
            run.log(f"  ⚠️ 디렉터리 목록도 못 읽었다: {type(e).__name__} {e}")
    return exts

PROBE = []
for e in list_exts(BUT_RECS[0]):
    try:
        a = wfdb.rdann(BUT_RECS[0], e, pn_dir=BUT_DIR)
        PROBE.append((e, len(a.sample), sorted(set(a.symbol))))
    except Exception:
        pass
if not PROBE:
    raise AssetError(f"{BUT_RECS[0]}: 읽히는 주석 파일이 하나도 없다")
EXT_P = next((e for e, _, _ in PROBE if e.lower().startswith("p")), None)
_rest = [(e, n_, sy) for e, n_, sy in PROBE if e != EXT_P]
EXT_Q = max(_rest, key=lambda t: (len(set(t[2]) & BEAT_SYM), t[1]))[0] if _rest else None
if EXT_P is None or EXT_Q is None:
    raise AssetError(f"주석 역할을 못 가렸다 — {[(e, n) for e, n, _ in PROBE]}")
run.log(f"  주석 확장자 — QRS `{EXT_Q}` · P `{EXT_P}` (**알아낸 값**)")

BUT, T0 = {}, time.time()
for rid in BUT_RECS:
    rec = wfdb.rdrecord(rid, pn_dir=BUT_DIR)
    sig = np.nan_to_num(np.asarray(rec.p_signal, float), nan=0.0, posinf=0.0, neginf=0.0)
    if sig.ndim != 2 or sig.shape[1] < 2:
        raise AssetError(f"{rid}: 2유도가 아니다 {sig.shape}")
    rp = np.asarray(wfdb.rdann(rid, EXT_Q, pn_dir=BUT_DIR).sample, int)
    pp = np.asarray(wfdb.rdann(rid, EXT_P, pn_dir=BUT_DIR).sample, int)
    if len(rp) < 5 or len(pp) < MIN_P:
        continue
    BUT[rid] = dict(sig=sig[:, :2], fs=int(rec.fs), r=rp, p=pp)
_nq = sum(len(v["r"]) for v in BUT.values()); _np_ = sum(len(v["p"]) for v in BUT.values())
run.log(f"  적재 {len(BUT)}개 · {time.time()-T0:.0f}초 · QRS **{_nq:,}** · P **{_np_:,}**")
_fs = sorted({v["fs"] for v in BUT.values()})
run.log(f"  ⚠️ **표본율 전이 가정** — BUT PDB {_fs}Hz vs SVDB 원본 **128Hz**(→360 보간).")
run.log("     `p_score` 는 위치 민감한 통계량이라 해상도가 다르면 감쇠도 달라진다.")
run.log("     λ 는 **BUT PDB 해상도에서 잰 값**이고, SVDB 로 옮길 때 이 가정이 붙는다.")
run.log("     SVDB 쪽 해상도가 더 거칠므로 실제 감쇠는 **더 심할** 가능성이 크고,")
run.log("     그러면 V1 의 상한은 **보수적**이 된다 — 판정 방향과 어긋나지 않는다")
run.log(f"  (문헌: QRS 7,638 · P 5,437 · P 없는 QRS 2,201(28.8%))")
CONFIG["but"] = dict(n_rec=len(BUT), n_qrs=int(_nq), n_p=int(_np_),
                     ann_ext=dict(qrs=EXT_Q, pwave=EXT_P))
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【V-A】 V0 재현 증명 — Q7-U 의 자를 그대로 다시 세운다
run.log("\n" + "=" * 100)
run.log("【V-A】 V0 — Q7-U `dwt|raw` 재현 증명 (R35 ⑦)")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

def ms2s(ms, fs):
    return int(round(ms * fs / 1000.0))

def _detrend(v):
    n = len(v)
    if n < 3:
        return np.asarray(v, float)
    t = np.arange(n, dtype=float)
    a, b = np.polyfit(t, np.asarray(v, float), 1)
    return np.asarray(v, float) - (a * t + b)

def detect_dwt(x, rp, fs):
    """NeuroKit2 이산 웨이블릿 구획(Martinez 2004 계열) — Q7-U 와 동일."""
    _, w = nk.ecg_delineate(x, rpeaks=rp, sampling_rate=fs, method=DELIN)
    p = w.get("ECG_P_Peaks", [])
    return np.asarray(sorted({int(v) for v in p
                              if v is not None and np.isfinite(v) and 0 <= v < len(x)}), int)

def score_win(x, pos, fs, half_ms):
    """★ **위치 중심 ±half_ms** 국소 두드러짐 — Q7-P0 의 `score_at` 정의(픽스처 ⑬)."""
    w = ms2s(half_ms, fs)
    out = []
    for q in np.atleast_1d(pos):
        q = int(q)
        a, b = max(q - w, 0), min(q + w + 1, len(x))
        if b - a < 3:
            out.append(0.0); continue
        seg = _detrend(x[a:b])
        m = float(np.median(np.abs(seg - np.median(seg)))) + 1e-12
        out.append(float(abs(seg[q - a]) / m))
    return np.asarray(out, float)

def f1_of(se, pp):
    return 0.0 if (se + pp) <= 0 else 2.0 * se * pp / (se + pp)

RIDS = sorted(BUT)
DET, T1_ = {}, time.time()
for rid in RIDS:
    d = BUT[rid]
    x = d["sig"][:, 0]
    try:
        pos = detect_dwt(x, d["r"], d["fs"])
    except Exception as e:
        run.log(f"    ⚠️ {rid} 실패: {type(e).__name__} {e}"); continue
    DET[rid] = dict(pos=pos, sc=score_win(x, pos, d["fs"], SCORE_HALF_MS))
run.log(f"  ({time.time()-T1_:.0f}초) 검출 완료 {len(DET)}/{len(RIDS)} 레코드")
if len(DET) < 10:
    raise AssetError(f"검출이 선 레코드가 {len(DET)}개 — 자를 세울 수 없다")

# 발화율 1.00 = 기권 없음 → 문턱 −inf (Q7-U 의 `loro_score_thr` 와 동치)
U1 = {}
for rid in DET:
    d = BUT[rid]; tol = TOL_MS * d["fs"] / 1000.0
    m, err = match_1d(DET[rid]["pos"], d["p"], tol)
    se = m / max(len(d["p"]), 1); pp = m / max(len(DET[rid]["pos"]), 1)
    U1[rid] = dict(se=se, pp=pp, f1=f1_of(se, pp),
                   err_ms=(err / d["fs"] * 1000.0))
sm, slo, shi, sn = boot_mean([U1[r]["se"] for r in U1], SEED0 + 11)
pm, plo, phi, _  = boot_mean([U1[r]["pp"] for r in U1], SEED0 + 12)
fm, flo, fhi, _  = boot_mean([U1[r]["f1"] for r in U1], SEED0 + 13)
run.log(f"\n  {'지표':<6}{'Q7-U':>9}{'재현':>9}{'Δ':>9}   CI")
rows = (("Se", REF_U1["se"], sm, slo, shi), ("PPV", REF_U1["pp"], pm, plo, phi),
        ("F1", REF_U1["f1"], fm, flo, fhi))
bad = []
for nm, ref_, got, lo_, hi_ in rows:
    dd = abs(got - ref_)
    tag = "✅" if dd <= TOL_U_WARN else ("⚠️" if dd <= TOL_U_FAIL else "❌")
    if dd > TOL_U_FAIL: bad.append((nm, ref_, got))
    run.log(f"  {nm:<6}{ref_:>9.4f}{got:>9.4f}{got-ref_:>+9.4f}   "
            f"[{lo_:.4f}, {hi_:.4f}]  {tag}")
CONFIG["V0"] = dict(se=sm, pp=pm, f1=fm, n=int(sn), ref=REF_U1, n_bad=len(bad))
run.save_json("config", CONFIG)
if bad:
    g_("V0", "❌ 기각", f"{len(bad)}개 지표가 {TOL_U_FAIL} 를 넘는다 — {[b[0] for b in bad]}")
    raise AssetError(
        "V0 재현 실패 — Q7-U 의 자를 다시 세우지 못했다. 라이브러리 버전이나 자산이 "
        "바뀐 것이고, 그러면 **감사 대상이 Q7-S′ 가 쓴 자가 아니다**. "
        f"기준 {REF_U1} vs 재현 se={sm:.4f} pp={pm:.4f} f1={fm:.4f}")
g_("V0", "✅ 지지", f"Q7-U 의 자를 다시 세웠다 — Se {sm:.4f} · PPV {pm:.4f} · F1 {fm:.4f}")
run.log(f"  ▸ **PPV {pm:.4f} = 발화의 {100*(1-pm):.1f}% 가 틀린 자리**다. 검출기는 기권하지")
run.log("    않으므로 그 값이 그대로 `p_score`·`pr_q8` 가 됐다 — 이게 감사 대상이다")

In [ ]:
# CELL 4 — 【V-B】 ★★★ V1 주 관문 — 감쇠 계수 λ 와 **상한**
run.log("\n" + "=" * 100)
run.log("【V-B】 V1(주) — 같은 비트에서 특징을 **두 번** 재어 감쇠 λ 를 실측한다")
run.log("=" * 100)
run.log("  ▸ 정답 P 가 **있는** 비트만 대상이다(특징이 기술하려는 모집단)")
run.log("  ▸ 검출기가 그 비트에 대해 낸 값을 쓴다 — **허용 오차 밖이어도 쓴다**.")
run.log("    그게 「조용한 오검출」이고, λ 가 잡아야 할 바로 그 성분이다")

def pair_true_obs(rid):
    """정답 P 마다 (참 PR, 관측 PR, 참 점수, 관측 점수, 적중여부)."""
    d = BUT[rid]; fs = d["fs"]; x = d["sig"][:, 0]
    tol = TOL_MS * fs / 1000.0
    lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)
    pos = DET[rid]["pos"]
    out = []
    for pt in d["p"]:
        # 이 정답 P 가 딸린 R — 뒤따르는 가장 가까운 QRS
        nxt = d["r"][d["r"] > pt]
        if not len(nxt):
            continue
        R = int(nxt[0])
        if not (R + lo <= pt < R + hi):        # 탐색창 밖이면 자가 볼 수 없는 P 다
            continue
        cand = pos[(pos >= R + lo) & (pos < R + hi)]
        if not len(cand):
            continue                            # 미발화 — λ 가 아니라 `p_miss` 의 몫
        q = int(cand[np.argmin(np.abs(cand - (R + lo + hi) // 2))])
        out.append((float((R - pt) / fs * 1000.0), float((R - q) / fs * 1000.0),
                    float(score_win(x, [pt], fs, SCORE_HALF_MS)[0]),
                    float(score_win(x, [q], fs, SCORE_HALF_MS)[0]),
                    bool(abs(q - pt) <= tol)))
    return out

LAM, T2_ = {"pr": [], "sc": [], "pr_hit": [], "sc_hit": []}, time.time()
NPAIR, HITR = [], []
for rid in DET:
    pr = pair_true_obs(rid)
    if len(pr) < 20:
        continue
    prt = np.array([p[0] for p in pr], float); pro = np.array([p[1] for p in pr], float)
    sct = np.array([p[2] for p in pr], float); sco = np.array([p[3] for p in pr], float)
    hit = np.array([p[4] for p in pr], bool)
    NPAIR.append(len(pr)); HITR.append(float(hit.mean()))
    def _c(a, b):
        if len(a) < 10 or np.std(a) < 1e-9 or np.std(b) < 1e-9:
            return np.nan
        return float(np.corrcoef(a, b)[0, 1])
    LAM["pr"].append(_c(prt, pro)); LAM["sc"].append(_c(sct, sco))
    LAM["pr_hit"].append(_c(prt[hit], pro[hit])); LAM["sc_hit"].append(_c(sct[hit], sco[hit]))
run.log(f"  ({time.time()-T2_:.0f}초) λ 를 잰 레코드 {len(NPAIR)} · "
        f"레코드당 정답 P 중앙 {int(np.median(NPAIR)) if NPAIR else 0} · "
        f"적중률 중앙 {np.median(HITR) if HITR else float('nan'):.4f}")
run.log(f"\n  {'특징':<10}{'λ(전체)':>10}{'CI':>22}{'λ(적중만)':>12}   해석")
LAMS = {}
for key, nm in (("sc", "p_score"), ("pr", "pr")):
    m_, lo_, hi_, n_ = boot_mean(LAM[key], SEED0 + 21)
    hm, _, _, _ = boot_mean(LAM[key + "_hit"], SEED0 + 22)
    LAMS[nm] = dict(lam=m_, lo=lo_, hi=hi_, n=n_, lam_hit=hm)
    run.log(f"  {nm:<10}{m_:>10.4f}  [{lo_:>7.4f}, {hi_:>7.4f}]{hm:>12.4f}   "
            + ("오검출까지 포함한 **실효** 감쇠" if key == "sc" else "위치 축"))
run.log("  ▸ λ(전체) < λ(적중만) 인 간격이 곧 **조용한 오검출의 비용**이다")

# ── ★★★ 탈감쇠 — 상한
lam = LAMS["p_score"]
obs, nul, MDE_S3 = REF_S3["p_score"], REF_S3["null"], REF_S3["mde"]
def deatt(a_obs, a_null, l):
    """★ **null 을 d 공간의 오프셋으로 두고 신호분만** 탈감쇠한다.

    `d_sig = d(obs) − d(null)` 을 λ 로 나누고 다시 null 을 얹는다. 관측 AUROC 를
    통째로 나누면 라벨셔플 null(0.5090)까지 부풀려 상한이 낙관적으로 커진다.
    ★ λ ≤ 1 이므로 결과는 항상 관측값 이상 — **상한**이다(R36 ①)."""
    if not np.isfinite(l) or l <= 0.05:
        return float("nan")
    d_sig = auc_to_d(a_obs) - auc_to_d(a_null)
    return d_to_auc(auc_to_d(a_null) + d_sig / min(max(l, 1e-6), 1.0))
UP = dict(point=deatt(obs, nul, lam["lam"]),
          hi=deatt(obs, nul, max(lam["lo"], 1e-6)),     # λ 가 작을수록 상한이 커진다
          lo=deatt(obs, nul, min(lam["hi"], 1.0)))
exc = {k: (v - nul) for k, v in UP.items()}
run.log(f"\n  ★★★ 탈감쇠 상한 — Q7-S′ S3 `p_score` {obs:.4f} · null {nul:.4f} · "
        f"MDE {MDE_S3:.4f}")
run.log(f"    관측 초과       **{obs - nul:+.4f}**")
run.log(f"    λ = {lam['lam']:.4f} 로 탈감쇠 → AUROC {UP['point']:.4f} · "
        f"초과 **{exc['point']:+.4f}**")
run.log(f"    λ 의 CI 를 반영한 상한 범위  초과 [{exc['lo']:+.4f}, {exc['hi']:+.4f}]")
run.log(f"    비교 대상 MDE   {MDE_S3:.4f}")
best_up = exc["hi"] if np.isfinite(exc["hi"]) else exc["point"]
if not np.isfinite(best_up):
    g_("V1", "⛔ 측정 불가", "λ 를 못 쟀다(레코드·쌍 부족)")
else:
    reach = best_up > MDE_S3
    g_("V1", "❌ 기각" if not reach else "✅ 지지",
       (f"★★★ **자를 완벽히 고쳐도 상한 {best_up:+.4f} ≤ MDE {MDE_S3:.4f}** — "
        "자는 병목이 아니다" if not reach else
        f"★★★ 상한 {best_up:+.4f} > MDE {MDE_S3:.4f} — **자가 병목일 수 있다**"))
    if not reach:
        run.log("       ▸ 「자를 더 갈면 된다」는 길을 **닫는다**. 남은 선택은 코호트를")
        run.log("         키우거나 갈래를 접는 것뿐이다")
    else:
        run.log("       ▸ 데이터 수집 **전에** 분절기를 고친다. 지금 모으면 무딘 자로 모으는 것")
run.log("  ⚠️ 이건 **상한**이지 점추정이 아니다 — 탈감쇠는 항상 키우는 방향이다(R36 ①)")
CONFIG["V1"] = dict(lam=LAMS, upper=UP, excess=exc, obs=obs, null=nul, mde=MDE_S3)
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【V-C】 V3 오검출 해부 — 틀린 발화는 **어디에** 떨어지나
run.log("\n" + "=" * 100)
run.log("【V-C】 V3 — 「조용한 오검출」의 해부")
run.log("=" * 100)
MISLOC = []                    # (정답 대비 오차 ms, R 대비 위치 ms)
HITERR = []
for rid in DET:
    d = BUT[rid]; fs = d["fs"]; tol = TOL_MS * fs / 1000.0
    lo, hi = ms2s(P_LO_MS, fs), ms2s(P_HI_MS, fs)
    pos = DET[rid]["pos"]
    for pt in d["p"]:
        nxt = d["r"][d["r"] > pt]
        if not len(nxt):
            continue
        R = int(nxt[0])
        if not (R + lo <= pt < R + hi):
            continue
        cand = pos[(pos >= R + lo) & (pos < R + hi)]
        if not len(cand):
            continue
        q = int(cand[np.argmin(np.abs(cand - (R + lo + hi) // 2))])
        e = (q - pt) / fs * 1000.0
        (HITERR if abs(q - pt) <= tol else MISLOC).append(
            (float(e), float((q - R) / fs * 1000.0)))
if len(HITERR) >= 10:
    he = np.array([h[0] for h in HITERR])
    run.log(f"  적중({len(HITERR):,}개) 오차 — 중앙 {np.median(he):+.1f}ms · "
            f"IQR {np.percentile(he,25):+.1f}~{np.percentile(he,75):+.1f}ms · "
            f"MAD {np.median(np.abs(he-np.median(he))):.1f}ms")
V3 = dict(n_hit=len(HITERR), n_mis=len(MISLOC))
if len(MISLOC) >= 10:
    mr = np.array([m[1] for m in MISLOC])      # R 대비 위치(음수 = R 앞)
    me = np.array([m[0] for m in MISLOC])
    run.log(f"  오검출({len(MISLOC):,}개) — 정답 대비 {np.median(me):+.1f}ms · "
            f"R 대비 위치 중앙 {np.median(mr):+.1f}ms")
    # 해부 — 생리 구간으로 나눈다(R 기준 · 음수가 앞)
    ZONES = (("직전 T 자리 (R−278~−200ms)", -278.0, -200.0),
             ("P 정상 자리 (R−200~−100ms)", -200.0, -100.0),
             ("PR 분절 (R−100~−42ms)",      -100.0, -42.0))
    for nm, a, b in ZONES:
        f = float(((mr >= a) & (mr < b)).mean())
        V3[nm] = f
        run.log(f"    {nm:<28} {f:6.1%}")
    late = float((me > 0).mean())
    V3["late_frac"] = late
    run.log(f"    ▸ 오검출의 **{late:.1%}** 가 정답보다 **뒤쪽**(R 에 가까운 쪽)이다")
    run.log("      뒤쪽 우세면 QRS 개시·PR 분절을 P 로 집는 것이고, 앞쪽 우세면 직전 T 다")
else:
    run.log("  오검출 표본이 10 미만 — 해부 생략(추측하지 않는다)")
CONFIG["V3"] = V3
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【V-D】 SVDB 적재 + 특징 (★ **파형을 쓰지 않는다** — 1판이 여기서 죽었다)
import pandas as pd
from scipy.stats import norm
run.log("\n" + "=" * 100)
run.log("【V-D】 SVDB — Q7-S′ 와 **같은** 특징을 다시 만든다")
run.log("=" * 100)
run.log("  ★★ **1판은 비트 배열에서 `p_score` 를 다시 계산하려다 실패했다**(실측 corr")
run.log("     0.7122 · 중앙 |Δ| 0.0000). Q7-P0 는 점수를 **연속 신호**에서 위치 중심")
run.log("     ±100ms(=36샘플) 창으로 쟀는데, 비트 절단은 R−278ms 에서 시작한다 —")
run.log("     P 후보가 앞쪽(비트 idx < 36)이면 창 왼쪽이 **비트 밖**이다. P 위치 중앙이")
run.log("     idx 41 · IQR 31~50 이라 하위 1/4쯤이 어긋났다(뒤쪽은 정확히 일치해서")
run.log("     |Δ| 중앙이 0 이었다). **비트 배열로는 그 자를 재현할 수 없다.**")
run.log("  ▸ 직전 비트로 왼쪽 문맥을 복원하는 방법도 `36 ≤ pre_rr ≤ 300` 샘플")
run.log("    (HR ≥ 72bpm)일 때만 되므로 느린 비트에서 또 구멍이 난다.")
run.log("  ▸ → **점수를 파형에서 다시 재지 않고 실측 분포로 옮긴다**(다음 셀).")
run.log("    BUT PDB 에서 이미 재고 있는 게 정확히 「정답 위치 점수 ↔ 검출 위치 점수」의")
run.log("    결합분포다. 파형이 필요 없고, 표본율·진폭 차이에도 불변이다")
for p_, why in ((SV5, "svdb_labels.py build"), (PDEL, "Q7-P0")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}(R16)")
D5 = np.load(SV5, allow_pickle=True); PD = np.load(PDEL, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); SYM = np.asarray(D5["sym"]).astype(str)
Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
# ★ `beat` 는 **읽지 않는다** — npz 는 지연 적재라 안 건드리면 메모리에 안 올라온다
if int((np.asarray(PD["pid"]).astype(int) != PID).sum()) or \
   int((np.asarray(PD["sym"]).astype(str) != SYM).sum()):
    raise AssetError("정합 깨짐 — Q7-P0 를 다시 돌린다")
P_IDX = np.asarray(PD["p_idx"]).astype(int); P_SC = np.asarray(PD["p_score"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
pidx0 = P_IDX[K].copy(); psc0 = P_SC[K].copy()
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
run.log(f"\n  비트 {len(K):,} · 레코드 {len(RS)} · 리듬 기저 재구성 완료 · "
        f"발화 {float((pidx0 >= 0).mean()):.4f}")
CONFIG["svdb"] = dict(n=int(len(K)), n_rec=int(len(RS)),
                      fire_rate=float((pidx0 >= 0).mean()), uses_waveform=False)
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【V-E】 ★★ V2 용량-반응(점수 영역 수송) + V4 영점 · **k=0 은 항등**
run.log("\n" + "=" * 100)
run.log("【V-E】 V2 — 실측 오차를 SVDB 에 **더 얹는다** (k=0,1,2)")
run.log("=" * 100)

# ── 수송 모형은 **BUT PDB 실측**에서만 온다 (SVDB 라벨을 안 본다 · R22)
def rank_z(v, groups):
    """★ **레코드 안 순위 → 정규 점수.** 코호트·표본율·진폭 차이에 불변이다.
    BUT PDB(원 표본율)와 SVDB(128→360 보간)의 점수 눈금이 다르므로,
    raw 값을 그대로 옮기면 안 된다."""
    z = np.full(len(v), np.nan)
    for u in np.unique(groups):
        m = np.where(groups == u)[0]
        x = np.asarray(v, float)[m]
        r = x.argsort().argsort().astype(float) + 1.0
        z[m] = norm.ppf(r / (len(x) + 1.0))
    return z

# BUT PDB — 정답 위치 점수 vs 검출 위치 점수의 결합분포(레코드별 순위 공간)
BT, BO, BR = [], [], []
for rid in DET:
    pr = pair_true_obs(rid)
    if len(pr) < 20:
        continue
    BT.append(np.array([p[2] for p in pr], float))
    BO.append(np.array([p[3] for p in pr], float))
    BR.append(np.full(len(pr), rid))
if not BT:
    raise AssetError("수송 모형을 세울 BUT PDB 쌍이 없다")
bt = np.concatenate(BT); bo = np.concatenate(BO); br = np.concatenate(BR)
zt = rank_z(bt, br); zo = rank_z(bo, br)
A_Z = float(np.nanmean([np.corrcoef(zt[br == u], zo[br == u])[0, 1]
                        for u in np.unique(br)
                        if np.std(zt[br == u]) > 0 and np.std(zo[br == u]) > 0]))
E_Z = zo - A_Z * zt                                   # 잔차 — 재표집 풀
E_Z = E_Z[np.isfinite(E_Z)]
P_MISS_EXTRA = float(1.0 - np.mean([U1[r]["se"] for r in U1]))
P_WRONG      = float(1.0 - np.mean([U1[r]["pp"] for r in U1]))
JIT = np.array([h[0] for h in HITERR], float)
WRO = np.array([m[0] for m in MISLOC], float)
if len(JIT) < 50 or len(WRO) < 20 or len(E_Z) < 200:
    raise AssetError(f"주입 분포가 너무 얇다 — 적중 {len(JIT)} · 오검출 {len(WRO)} · "
                     f"수송 잔차 {len(E_Z)}")
run.log(f"  수송 모형(BUT PDB 실측 · 순위 공간) — 기울기 a **{A_Z:.4f}** · "
        f"잔차 SD {np.std(E_Z):.4f} (n={len(E_Z):,})")
run.log(f"  위치 주입 — 놓침 {P_MISS_EXTRA:.4f} · 오검출 {P_WRONG:.4f} · "
        f"적중 지터 IQR {np.percentile(JIT,25):+.1f}~{np.percentile(JIT,75):+.1f}ms")
run.log(f"  ▸ V1 의 λ(raw 공간) {CONFIG['V1']['lam']['p_score']['lam']:.4f} vs "
        f"a(순위 공간) {A_Z:.4f} — 크게 다르면 비선형이 있다는 뜻")
run.log("  ⚠️ **V1 과 V2 는 같은 실측 쌍 분포를 공유한다.** 그래서 「가정이 다른 두 경로」가")
run.log("     아니라 **「닫힌형 이론 상한」 vs 「그 분포를 실제 추정량에 통과시킨 재표집」**")
run.log("     이다. V2 는 리듬 잔차화·레코드 구조·매칭까지 다 거치므로 여전히 다른 확인이지만,")
run.log("     독립 증거로 인용하면 안 된다")

def transport(sc, rid, rng):
    """★ 레코드 안 **순위 공간**에서 `z\' = a·z + e` 를 걸고, **같은 집합의 경험분위수**로
    되돌린다.

    ★★ 왜 집합 내부에서 자기완결로 하나: 용량 k 가 올라가면 발화 집합이 줄어드는데,
      바깥에 고정해 둔 분위수표를 쓰면 길이가 어긋난다. 1판이 그렇게 짜서
      **항등 검사가 |Δ| 1.29 로 잡아냈다**(순위 r/(m+1) 과 되돌리기 q·(m−1) 불일치).
    ★ `a=1 · e=0` 이면 순위가 그대로라 **원값이 정확히 복원**된다 → k=0 항등이
      **구성으로** 보장된다.
    ★ 값의 눈금(코호트·표본율·진폭)에 **불변**이다 — BUT PDB 와 SVDB 는 눈금이 다르다."""
    out = np.array(sc, float).copy()
    for u in np.unique(rid):
        m = np.where(rid == u)[0]
        x = out[m]; mm = len(x)
        if mm < 5:
            continue
        r = x.argsort().argsort().astype(float) + 1.0
        z = norm.ppf(r / (mm + 1.0))
        zp = A_Z * z + rng.choice(E_Z, mm)
        rp = np.clip(np.round(norm.cdf(zp) * (mm + 1.0)).astype(int), 1, mm)
        out[m] = np.sort(x)[rp - 1]
    return out

P_LO_S, P_HI_S = int(round(P_LO_MS * FS / 1000.0)), int(round(P_HI_MS * FS / 1000.0))
LO_I, HI_I = RPRE + P_LO_S, RPRE + P_HI_S

def corrupt(pidx, psc, rng):
    """★ 오차를 **한 번 더** 얹는다. 파라미터는 전부 BUT PDB 실측이고 **라벨을 안 본다**.
    점수는 순위 공간 수송, 위치는 실측 변위 주입."""
    p = pidx.copy(); s = psc.copy()
    okv = np.where(p >= 0)[0]
    n_ms = np.zeros(len(okv))
    w = rng.random_sample(len(okv)) < P_WRONG
    n_ms[w] = rng.choice(WRO, int(w.sum()))
    n_ms[~w] = rng.choice(JIT, int((~w).sum()))
    p[okv] = np.clip(p[okv] + np.round(n_ms * FS / 1000.0).astype(int), LO_I, HI_I - 1)
    s[okv] = transport(s[okv], RID[okv], rng)
    miss = okv[rng.random_sample(len(okv)) < P_MISS_EXTRA]
    p[miss] = -1; s[miss] = 0.0                        # Q7-P0 규약 — 미발화는 점수 0
    return p, s

def basis_ext(idx):
    return np.c_[np.ones(len(idx)), f1[idx], f3[idx], f4[idx],
                 np.column_stack([f2[k][idx] for k in FULL_K])]

def resid(v, idx):
    X = basis_ext(idx); y = v[idx]
    okm = np.isfinite(y)
    if okm.sum() < X.shape[1] + 5:
        return np.full(len(idx), np.nan)
    b = np.linalg.lstsq(X[okm], y[okm], rcond=None)[0]
    return y - X @ b

def matched_auc(vsub, idx):
    tt = TT[idx]; key = np.round(f1[idx]).astype(int)
    win = tie = tot = 0.0
    for kk in np.unique(key):
        m = np.where(key == kk)[0]
        a = vsub[m[tt[m]]]; b = vsub[m[~tt[m]]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if not len(a) or not len(b):
            continue
        d = a[:, None] - b[None, :]
        win += float((d > 0).sum()); tie += float((d == 0).sum()); tot += float(d.size)
    return (win + 0.5 * tie) / tot if tot >= MIN_PAIR else float("nan")

REC_OK = [r for r in RS
          if (lambda i: TT[i].sum() >= MIN_S and (~TT[i]).sum() >= MIN_N)(np.where(RID == r)[0])]

def s3_of(vec):
    per = []
    for r in REC_OK:
        idx = np.where(RID == r)[0]
        a = matched_auc(resid(vec, idx), idx)
        if np.isfinite(a):
            per.append(a)
    return boot_mean(per, SEED0 + 61)

# ── ★★ 수송 항등 검사 — a=1 · e=0 이면 원값이 나와야 한다(구현 증명)
_A, _E = A_Z, E_Z
A_Z, E_Z = 1.0, np.zeros(1)
_fr = pidx0 >= 0
_id = transport(psc0[_fr], RID[_fr], np.random.RandomState(0))
A_Z, E_Z = _A, _E
_dev = float(np.nanmax(np.abs(_id - psc0[_fr])))
run.log(f"\n  ★ 수송 항등 검사 — a=1·e=0 에서 최대 |Δ| **{_dev:.2e}**")
if _dev > 1e-9:
    raise AssetError(
        f"수송 사상이 항등이 아니다(최대 |Δ| {_dev:.3e}) — 순위/분위수 되돌리기가 틀렸다. "
        "이 상태로는 k=1,2 가 무엇을 재는지 알 수 없다")

run.log(f"\n  {'k':<4}{'AUROC':>10}{'CI':>22}{'초과':>10}{'n':>6}")
V2, T3_ = {}, time.time()
for k in DOSES:
    if k == 0:
        m_, lo_, hi_, n_ = s3_of(psc0)               # ★ 주입 경로를 아예 안 거친다
        _sd = 0.0
    else:
        reps = []
        for rep in range(N_REP):
            rng = np.random.RandomState(SEED0 + 900 + 17 * k + rep)
            p, sc = pidx0.copy(), psc0.copy()
            for _ in range(k):
                p, sc = corrupt(p, sc, rng)
            reps.append(s3_of(sc)[0])
        m_ = float(np.nanmean(reps)); _sd = float(np.nanstd(reps))
        lo_ = hi_ = float("nan"); n_ = len(REC_OK)
    V2[k] = dict(auc=m_, lo=lo_, hi=hi_, n=int(n_), sd=_sd,
                 excess=float(m_ - REF_S3["null"]))
    run.log(f"  {k:<4}{m_:>10.4f}" +
            (f"  [{lo_:>7.4f}, {hi_:>7.4f}]" if np.isfinite(lo_) else " " * 22) +
            f"{m_-REF_S3['null']:>+10.4f}{n_:>6}" +
            (f"   (반복 {N_REP}회 SD {_sd:.4f})" if k else "   ← **항등**"))
run.log(f"  ({time.time()-T3_:.0f}초)")

# ── ★★ 항등 대조 — k=0 이 Q7-S′ 와 같아야 한다
d0 = abs(V2[0]["auc"] - REF_S3["p_score"])
run.log(f"\n  ★★ 항등 대조 — k=0 {V2[0]['auc']:.4f} vs Q7-S′ {REF_S3['p_score']:.4f} · "
        f"|Δ| {d0:.6f} (허용 {IDENT_TOL})")
if d0 > IDENT_TOL:
    g_("V2", "❌ 기각", f"**k=0 이 항등이 아니다** — |Δ| {d0:.6f}")
    raise AssetError(
        f"항등 대조 실패 — 오차를 0번 얹었는데 Q7-S′ 의 S3({REF_S3['p_score']:.4f})가 "
        f"안 나온다({V2[0]['auc']:.4f}). 특징 재구성이 틀렸고, k=1,2 의 기울기는 "
        "아무것도 뜻하지 않는다")
slope = V2[1]["excess"] - V2[0]["excess"] if 1 in V2 else float("nan")
extrap = V2[0]["excess"] - slope
run.log(f"  기울기(용량 1당) **{slope:+.4f}** · k=−1 외삽 초과 **{extrap:+.4f}**")
up1 = CONFIG["V1"]["excess"]["hi"]
agree = (np.isfinite(extrap) and np.isfinite(up1) and extrap > 0 and up1 > 0
         and 0.5 <= extrap / up1 <= 2.0)
g_("V2", "✅ 지지" if agree else "⚠️ 미결",
   (f"두 계산이 일치 — V2 외삽 {extrap:+.4f} vs V1 상한 {up1:+.4f}" if agree else
    f"★ 두 계산이 어긋난다 — V2 외삽 {extrap:+.4f} vs V1 상한 {up1:+.4f}. "
    "**어느 쪽도 점추정으로 인용하지 않는다**"))

# ── V4 영점 — 같은 수송을 **무정보 점수**에 걸면 0.5 를 유지하나
run.log("\n  V4 — 영점(무정보 점수에 같은 수송)")
rng0 = np.random.RandomState(SEED0 + 777)
sc_rand = np.where(pidx0 >= 0, rng0.permutation(psc0), 0.0)   # 라벨 관계만 끊는다
z0 = s3_of(sc_rand)
rngz = np.random.RandomState(SEED0 + 778)
_p, sc_z = corrupt(pidx0.copy(), sc_rand.copy(), rngz)
z1 = s3_of(sc_z)
run.log(f"    무정보 k=0  {z0[0]:.4f} [{z0[1]:.4f}, {z0[2]:.4f}]")
run.log(f"    무정보 k=1  {z1[0]:.4f} [{z1[1]:.4f}, {z1[2]:.4f}]")
z_ok = all(abs(z[0] - 0.5) <= max(mde(z[1], z[2]), 0.01) for z in (z0, z1))
g_("V4", "✅ 지지" if z_ok else "❌ 기각",
   "수송이 인공 신호를 만들지 않는다" if z_ok else
   "★ **수송 자체가 신호를 만든다** — V2 의 기울기를 해석할 수 없다")
CONFIG["V2"] = {str(k): v for k, v in V2.items()}
CONFIG["V2_slope"] = dict(slope=float(slope), extrap=float(extrap), agree=bool(agree))
CONFIG["V4"] = dict(k0=list(z0[:3]), k1=list(z1[:3]), ok=bool(z_ok))
CONFIG["inject"] = dict(p_miss=P_MISS_EXTRA, p_wrong=P_WRONG, a_z=A_Z,
                        e_sd=float(np.std(E_Z)), jit_n=int(len(JIT)),
                        wro_n=int(len(WRO)), e_n=int(len(E_Z)))
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【V-F】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

# ① 오차 분포 — 적중 지터 vs 오검출 변위
ax[0].hist(JIT, bins=40, alpha=.7, label=f"hit jitter (n={len(JIT):,})", color="tab:blue")
ax[0].hist(WRO, bins=40, alpha=.55, label=f"misdetection (n={len(WRO):,})", color="tab:red")
ax[0].axvline(0, color="k", lw=.9)
ax[0].axvspan(-TOL_MS, TOL_MS, color="tab:green", alpha=.10)
ax[0].set_xlabel("detected - true P  (ms)   [green = +-50ms tolerance]")
ax[0].set_ylabel("count"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

# ② ★★★ 감쇠 상한 vs MDE
lbl2 = ["observed excess", "de-attenuated (point)", "de-attenuated (upper)"]
v2 = [obs - nul, exc["point"], exc["hi"]]
ax[1].barh(np.arange(3), v2, color=["tab:blue", "tab:orange", "tab:red"])
ax[1].axvline(MDE_S3, ls="--", color="k", lw=1.2)
ax[1].annotate(f"MDE {MDE_S3:.4f}", (MDE_S3, 2.4), fontsize=7, rotation=90, ha="right")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(3)); ax[1].set_yticklabels(lbl2, fontsize=8)
ax[1].set_xlabel(f"V1 : S3 excess over null (lambda={lam['lam']:.3f})")
ax[1].grid(alpha=.3, axis="x")

# ③ 용량-반응
ks = sorted(V2)
ax[2].plot(ks, [V2[k]["excess"] for k in ks], "o-", color="tab:red", label="p_score excess")
if np.isfinite(extrap):
    ax[2].plot([-1], [extrap], "*", ms=13, color="tab:orange", label="k=-1 extrapolation")
    ax[2].plot([-1, 0], [extrap, V2[0]["excess"]], ":", color="tab:orange")
ax[2].axhline(0, color="k", lw=.9)
ax[2].axhline(MDE_S3, ls="--", color="k", lw=.9)
ax[2].annotate("MDE", (max(ks), MDE_S3), fontsize=7, va="bottom", ha="right")
ax[2].set_xticks([-1] + ks)
ax[2].set_xlabel("extra doses of measured detector error (k)")
ax[2].set_ylabel("S3 excess over null")
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q7v_ruler_audit", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
for g in ("V0", "V1", "V2", "V4"):
    run.log(f"  {g:<4}{VERD.get(g, '(미실행)')}")
run.log("")
if not ok_("V0"):
    run.log("  ⛔ 자를 재현 못 했다 — 아래 어떤 문장도 쓰지 않는다")
elif not ok_("V4"):
    run.log("  ⛔ 주입이 인공 신호를 만든다 — V2 의 기울기를 해석할 수 없다")
elif no_("V1"):
    run.log("  ★★★ **자는 병목이 아니다.** 감쇠를 완전히 되돌려도 상한이 MDE 밑이다.")
    run.log("      → 「분절기를 더 갈면 된다」는 길을 **닫는다**.")
    run.log("      → 남은 선택은 둘뿐이다 — **코호트를 키우거나(필요 표본은 Q7-S″ 의 W2),**")
    run.log("        **형태 갈래를 접고 리듬-only 보정(Q3·Q4)으로 간다**")
elif ok_("V1"):
    run.log("  ★★★ **자가 병목일 수 있다.** 감쇠를 되돌리면 상한이 MDE 를 넘는다.")
    run.log("      → **데이터를 모으기 전에 분절기를 고친다.** 지금 모으면 무딘 자로 모으는")
    run.log("        것이고, 표본이 커져도 감쇠는 그대로다(감쇠는 n 으로 안 줄어든다)")
    run.log(f"      ⚠️ **실행 후 정정**: 1판은 「다음 표적 = 오검출 {100*P_WRONG:.1f}% 를 줄이는 것」")
    run.log("        이라고 찍었는데 **틀렸다**. 산수를 맞춰 보면 1−PPV 는 **위치 오류가 아니라**")
    run.log("        **P 가 없는 비트에 기권 없이 쏜 것**이다(P 부재율 0.288 과 일치하고,")
    run.log("        진짜 P 가 있는 비트의 적중률은 0.99 다). 오검출을 0 으로 만들어도 λ 는")
    run.log("        λ(적중만) 까지밖에 안 오른다 — **표적은 점수 통계량의 신뢰도**다")
else:
    run.log("  ⚠️ 미결 — λ 를 못 쟀거나 표본이 얇다. 상한을 인용하지 않는다")
# ⚠️ **실행 후 정정**(2026-08-04) — 1판 요약이 여기서 「가정이 다른 두 경로」라고 찍었는데
#    같은 셀 위쪽 주석과 **정반대**였다. V1·V2 는 **같은 실측 쌍 분포를 공유**한다.
#    숫자는 그대로이고 이 문장만 고쳤다(외부 검토 3🟡② 지적).
run.log(f"\n  ▸ V1(닫힌형 이론 상한)과 V2(같은 분포를 실제 추정량에 통과시킨 재표집)는 "
        f"**같은 실측 쌍 분포를 공유**한다 — {'일치' if agree else '**어긋난다**'}")
if not agree:
    run.log("    ★ 이건 「독립 경로가 불일치」가 아니라 **「같은 재료를 두 방식으로 처리했더니**")
    run.log("      **다르게 나왔다」**이고, 그쪽이 더 나쁘다 — 계산 하나에 문제가 있다는 뜻이다")
run.log("  ▸ 어느 판정이든 이 런은 **SVEB 질문에 답하지 않는다** — 「더 모을 가치가 있나」")
run.log("    에만 답한다(R35 ①: 자를 먼저 세워라)")

run.finish({
    "exp_id": "quest46_q7v_ruler_audit",
    "metric": "deattenuated_s3_excess_upper",
    "value": float(exc["hi"]) if np.isfinite(exc["hi"]) else float("nan"),
    "passed": bool(ok_("V0") and ok_("V4")),
    "summary": ("자가 병목인가 — BUT PDB 정답으로 감쇠 계수를 실측해 S3 의 탈감쇠 상한을 "
                "내고, 같은 오차를 SVDB 에 더 얹어 용량-반응을 잰다. 상한이 MDE 밑이면 "
                "자를 고쳐도 못 넘는다."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "V0": CONFIG.get("V0", {}), "V1": CONFIG.get("V1", {}), "V2": CONFIG.get("V2", {}),
    "V2_slope": CONFIG.get("V2_slope", {}), "V3": CONFIG.get("V3", {}),
    "V4": CONFIG.get("V4", {}), "inject": CONFIG.get("inject", {}),
    "but": CONFIG.get("but", {}), "svdb": CONFIG.get("svdb", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step ruler-audit`")